Setup

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import sys # Import the sys module to get Python version

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ Python: {sys.version.split()[0]}") # Use sys.version to get Python version

✅ TensorFlow: 2.20.0
✅ Python: 3.12.13


In [2]:
# 🔧 Install dependencies (first run only)
# !pip install tensorflow_datasets librosa matplotlib scikit-learn

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import librosa
import os
from sklearn.model_selection import train_test_split

# 🎨 Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
tf.get_logger().setLevel('ERROR')  # Reduce noise

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ Librosa: {librosa.__version__}")

✅ TensorFlow: 2.20.0
✅ Librosa: 0.11.0


Load Google Speech Commands v2 (Yes/No Only)

In [ ]:
# ↓ Cell 2 (REPLACED): Load Speech Commands v2 using TensorFlow Datasets
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import os
from sklearn.model_selection import train_test_split

def load_speech_commands_tfds(batch_size=32, sample_rate=16000, max_duration=1.0):
    """Load Yes/No samples from Speech Commands v0.02 using TFDS"""

    # Load full dataset (cached after first download ~2GB)
    ds, info = tfds.load(
        'speech_commands:0.0.3',
        split='train',
        with_info=True,
        shuffle_files=True
    )

    # Filter to Yes/No only (+ silence for negative examples)
    def filter_yes_no_silence(example):
        label = example['label']
        # Labels: 0='silence', 1='unknown', 2='yes', 3='no', ...
        return tf.logical_or(
            tf.logical_or(label == 0, label == 2),  # silence or yes
            label == 3  # or no
        )

    ds_filtered = ds.filter(filter_yes_no_silence)

    # Preprocess: truncate/pad to fixed length, normalize
    def preprocess(example):
        audio = example['audio']
        label = example['label']

        # Convert to float32, normalize to [-1, 1]
        audio = tf.cast(audio, tf.float32) / 32768.0

        # Truncate or pad to 1 second (16000 samples @ 16kHz)
        target_length = int(sample_rate * max_duration)
        if tf.shape(audio)[0] > target_length:
            audio = audio[:target_length]
        else:
            audio = tf.pad(audio, [[0, target_length - tf.shape(audio)[0]]])

        # Remap labels: silence(0)→0(No), yes(2)→1(Yes), no(3)→0(No)
        label_binary = tf.cond(
            label == 2,
            lambda: tf.constant(1, dtype=tf.int32),  # Yes → 1
            lambda: tf.constant(0, dtype=tf.int32)   # No/Silence → 0
        )

        return audio, label_binary

    ds_processed = ds_filtered.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # Convert TFDS dataset to numpy arrays for consistency with previous cells if needed,
    # or continue with TFDS dataset format for model training.
    # For this example, we'll convert to numpy for compatibility with the existing split logic.
    audios_list = []
    labels_list = []
    for audio, label in ds_processed.as_numpy_iterator():
        audios_list.append(audio)
        labels_list.append(label)

    X = np.array(audios_list, dtype=np.float32)
    y = np.array(labels_list, dtype=np.int32)

    return X, y

# ↻ Load data
print("↓️ Loading Google Speech Commands v2 (Yes/No subset) via TFDS...")
X, y = load_speech_commands_tfds(batch_size=32)

# Combine + shuffle (already shuffled by TFDS if configured, but keeping for consistency)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Loaded: {len(X)} samples | Yes: {np.sum(y==1)}, No: {np.sum(y==0)}")
print(f"📦 Train: {len(X_train)} | Val: {len(X_val)}")

↓️ Loading Google Speech Commands v2 (Yes/No subset) via TFDS...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Visualize Class Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import random

#  1. Class Distribution (Full Dataset)
print(" Dataset Summary:")
print(f"   Total samples: {len(y)}")
print(f"   ✅ Yes (1): {np.sum(y==1)} ({np.mean(y==1)*100:.1f}%)")
print(f"   ❌ No/Silence (0): {np.sum(y==0)} ({np.mean(y==0)*100:.1f}%)")
print(f"     Imbalance ratio: {np.sum(y==0)/np.sum(y==1):.2f}x")
print()

plt.figure(figsize=(6, 4))
labels = ['No/Silence (0)', 'Yes (1)']
counts = [np.sum(y==0), np.sum(y==1)]
colors = ['#ff6b6b', '#4ecdc4']
plt.bar(labels, counts, color=colors, edgecolor='black', alpha=0.9)
plt.title(' Class Distribution: Yes vs No/Silence')
plt.ylabel('Number of Samples')
for i, v in enumerate(counts):
    plt.text(i, v + 100, str(v), ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

# 🔊 2. Visualize 4 Sample Waveforms (2 Yes, 2 No)
print(" Sample Waveforms (1-second audio @ 16kHz):")
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.flatten()

# Find indices for Yes and No
yes_idx = np.where(y == 1)[0]
no_idx = np.where(y == 0)[0]

# Pick 2 random from each class
sample_indices = random.sample(list(yes_idx), 2) + random.sample(list(no_idx), 2)
random.shuffle(sample_indices)

for i, idx in enumerate(sample_indices):
    audio = X[idx]
    label = " Yes" if y[idx] == 1 else " No/Silence"

    # Plot waveform
    time_axis = np.linspace(0, 1, len(audio))  # 1 second
    axes[i].plot(time_axis, audio, linewidth=0.5, color=colors[y[idx]])
    axes[i].set_title(f"Sample #{idx+1} | {label}")
    axes[i].set_xlabel("Time (s)")
    axes[i].set_ylabel("Amplitude")
    axes[i].grid(alpha=0.3)
    axes[i].set_ylim(-1.2, 1.2)

plt.tight_layout()
plt.show()

#  3. MFCC Feature Preview (What the model actually sees)
print(" MFCC Features Preview (13 coefficients × 49 frames):")
print("   (This is the input shape for our MLP: [49, 13, 1])")

# Pick one Yes and one No sample
sample_yes = X[np.where(y==1)[0][0]]
sample_no = X[np.where(y==0)[0][0]]

def plot_mfcc(audio, title, cmap='viridis'):
    """Extract and plot MFCC spectrogram"""
    mfcc = librosa.feature.mfcc(
        y=audio, sr=16000, n_mfcc=13,
        n_fft=2048, hop_length=512  # ~25ms frame, 10ms hop
    )
    plt.figure(figsize=(8, 3))
    plt.imshow(mfcc, aspect='auto', origin='lower', cmap=cmap)
    plt.title(title)
    plt.xlabel("Time Frame (~49)")
    plt.ylabel("MFCC Coefficient (1-13)")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
    return mfcc

print("\n Example: 'Yes' sample → MFCC")
mfcc_yes = plot_mfcc(sample_yes, "MFCC: 'Yes' Keyword", cmap='plasma')

print("\n Example: 'No' sample → MFCC")
mfcc_no = plot_mfcc(sample_no, "MFCC: 'No' Keyword", cmap='viridis')

#  4. Input Shape Reminder for Model
print("\n Model Input Specification:")
print(f"   • Raw audio shape: {X[0].shape} → [16000] samples")
print(f"   • MFCC shape: {mfcc_yes.shape} → [13, 49]")
print(f"   • Model input (with channel): [49, 13, 1]")
print(f"   • Flatten → Dense layers → Binary output")

Extract & Normalize MFCC Features

In [ ]:

# Converts raw audio [N, 16000] → MFCC features [N, 49, 13, 1]
import librosa
from sklearn.preprocessing import StandardScaler
import numpy as np

print(" Extracting MFCC features...")

def audio_to_mfcc(audio, sr=16000, n_mfcc=13):
    # Standard KWS settings: frame_length=25ms, frame_step=10ms
    # n_fft=512, hop_length=320 → ~49 frames for 1s @ 16kHz
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc,
                                n_fft=512, hop_length=320)
    mfcc = mfcc.T  # [frames, n_mfcc]
    return np.expand_dims(mfcc, axis=-1)  # [49, 13, 1]

# Extract features (list comprehension for clarity & speed)
X_train_mfcc = np.array([audio_to_mfcc(a) for a in X_train])
X_val_mfcc = np.array([audio_to_mfcc(a) for a in X_val])

#  Normalize features (critical for NN convergence)
scaler = StandardScaler()
N_train, T, F, C = X_train_mfcc.shape
X_train_mfcc = scaler.fit_transform(X_train_mfcc.reshape(N_train, -1)).reshape(N_train, T, F, C)

N_val = X_val_mfcc.shape[0]
X_val_mfcc = scaler.transform(X_val_mfcc.reshape(N_val, -1)).reshape(N_val, T, F, C)

print(" Extraction complete!")
print(f"   Train: {X_train_mfcc.shape} | Val: {X_val_mfcc.shape}")
print(f"   Expected model input: (49, 13, 1)")

Build Baseline MLP Model

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

# Auto-detect input shape from data
INPUT_SHAPE = X_train_mfcc.shape[1:]  # (49, 13, 1)
NUM_CLASSES = 2

def build_baseline_mlp(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape, name='mfcc_input'),
        layers.Flatten(name='flatten_mfcc'),
        layers.Dense(32, activation='relu', name='dense_1'),
        layers.Dense(16, activation='relu', name='dense_2'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    return model

mlp_model = build_baseline_mlp(INPUT_SHAPE, NUM_CLASSES)
mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
mlp_model.summary()

Train Model + Live Diagnostics

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

#  Handle class imbalance (No: 67% vs Yes: 33%)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"  Class weights applied: {class_weight_dict}")

#  Callbacks for stable training
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
]

print(" Starting training...")
history = mlp_model.fit(
    X_train_mfcc, y_train,
    validation_data=(X_val_mfcc, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

#  Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Acc', marker='s')
plt.title(' Accuracy'); plt.legend(); plt.grid()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s')
plt.title(' Loss'); plt.legend(); plt.grid()
plt.tight_layout()
plt.show()

print(f"\n Final Val Accuracy: {history.history['val_accuracy'][-1]:.3f}")

Add Dropout Regularization (Fastest Fix)

In [ ]:
# 🔄 Cell 6a: Upgrade 1 — Dropout MLP + Training Curves
def build_mlp_dropout(input_shape, num_classes):
    return tf.keras.Sequential([
        layers.Input(shape=input_shape, name='mfcc_input'),
        layers.Flatten(name='flatten'),
        layers.Dense(32, activation='relu', name='dense_1'),
        layers.Dropout(0.3, name='drop_1'),
        layers.Dense(16, activation='relu', name='dense_2'),
        layers.Dropout(0.3, name='drop_2'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ])

print(" Building MLP + Dropout...")
model_drop = build_mlp_dropout(INPUT_SHAPE, NUM_CLASSES)
model_drop.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print(" Training with Dropout...")
history_drop = model_drop.fit(
    X_train_mfcc, y_train,
    validation_data=(X_val_mfcc, y_val),
    epochs=20, batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

#  Plot Dropout Training Curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_drop.history['accuracy'], label='Train Acc', marker='o', linewidth=2)
plt.plot(history_drop.history['val_accuracy'], label='Val Acc', marker='s', linewidth=2)
plt.title('📈 Dropout MLP: Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_drop.history['loss'], label='Train Loss', marker='o', linewidth=2)
plt.plot(history_drop.history['val_loss'], label='Val Loss', marker='s', linewidth=2)
plt.title('📉 Dropout MLP: Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n Dropout Val Accuracy: {max(history_drop.history['val_accuracy']):.3f}")
print(f" Params: {model_drop.count_params():,}")

In [ ]:
#  Cell 6b: Upgrade 2 — Shallow CNN + Training Curves
def build_shallow_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape, name='mfcc_input')
    x = layers.Conv2D(8, (3,3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='kws_shallow_cnn')

print(" Building Shallow CNN...")
model_cnn = build_shallow_cnn(INPUT_SHAPE, NUM_CLASSES)
model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()

print(" Training CNN...")
history_cnn = model_cnn.fit(
    X_train_mfcc, y_train,
    validation_data=(X_val_mfcc, y_val),
    epochs=15, batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

#  Plot CNN Training Curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Train Acc', marker='o', linewidth=2)
plt.plot(history_cnn.history['val_accuracy'], label='Val Acc', marker='s', linewidth=2)
plt.title('📈 Shallow CNN: Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Train Loss', marker='o', linewidth=2)
plt.plot(history_cnn.history['val_loss'], label='Val Loss', marker='s', linewidth=2)
plt.title(' Shallow CNN: Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n CNN Val Accuracy: {max(history_cnn.history['val_accuracy']):.3f}")
print(f" Params: {model_cnn.count_params():,}")

In [ ]:
# 📊 Cell 6c: Compare Baseline vs Upgraded Model
plt.figure(figsize=(12, 4))

# Accuracy comparison
plt.subplot(1, 2, 1)
plt.plot(history.history['val_accuracy'], label='Baseline MLP', marker='o', linewidth=2)
if 'history_drop' in locals():
    plt.plot(history_drop.history['val_accuracy'], label='MLP + Dropout', marker='s', linewidth=2)
elif 'history_cnn' in locals():
    plt.plot(history_cnn.history['val_accuracy'], label='Shallow CNN', marker='^', linewidth=2)
plt.title('📈 Validation Accuracy Comparison')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)

# Loss comparison
plt.subplot(1, 2, 2)
plt.plot(history.history['val_loss'], label='Baseline MLP', marker='o', linewidth=2)
if 'history_drop' in locals():
    plt.plot(history_drop.history['val_loss'], label='MLP + Dropout', marker='s', linewidth=2)
elif 'history_cnn' in locals():
    plt.plot(history_cnn.history['val_loss'], label='Shallow CNN', marker='^', linewidth=2)
plt.title('📉 Validation Loss Comparison')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 📏 Model size & params comparison
print("\n📦 Model Comparison:")
print(f"   Baseline MLP: {mlp_model.count_params():,} params | {os.path.getsize('kws_yesno_baseline_mlp.tflite')/1024:.1f} KB")
if 'model_drop' in locals():
    print(f"   MLP+Dropout:  {model_drop.count_params():,} params | ~{mlp_model.count_params()*4/1024:.1f} KB (int8)")
elif 'model_cnn' in locals():
    print(f"   Shallow CNN:  {model_cnn.count_params():,} params | ~{model_cnn.count_params()*4/1024:.1f} KB (int8)")

Export the WINNER (Dropout MLP)

In [ ]:

import os

def representative_dataset():
    for i in range(min(100, len(X_train_mfcc))):
        yield [X_train_mfcc[i:i+1].astype(np.float32)]

print(" Exporting the BEST model (Dropout MLP)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_drop)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
model_path = "kws_yesno_best_model.tflite"
with open(model_path, "wb") as f:
    f.write(tflite_model)

print(f" Saved: {model_path}")
print(f" Size: {os.path.getsize(model_path)/1024:.2f} KB")

Quantize & Export to TFLite Micro (B-U585I-IOT02A Ready)

In [ ]:
import os

def representative_dataset():
    # Use 100 samples from training set for int8 calibration
    for i in range(min(100, len(X_train_mfcc))):
        yield [X_train_mfcc[i:i+1].astype(np.float32)]

print(" Converting to TFLite (post-training quantization)...")
converter = tf.lite.TFLiteConverter.from_keras_model(mlp_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# Keep float32 I/O for Colab testing; X-CUBE-AI handles int8 buffer scaling on-device
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
model_path = "kws_yesno_baseline_mlp.tflite"
with open(model_path, "wb") as f:
    f.write(tflite_model)

print(f"\n Export complete!")
print(f" Model: {model_path}")
print(f" Size: {os.path.getsize(model_path) / 1024:.2f} KB (Target: <1500 KB)")
print(f" Params: {mlp_model.count_params():,} (Target: <100K)")